# Olist 데이터 탐색 (Data Exploration)

분석/스키마 설계에 앞서 각 CSV 파일의 컬럼, 데이터 타입, 행 개수, 샘플 데이터를 확인한다.

In [1]:
import glob
import duckdb
import pandas as pd

con = duckdb.connect()  # in-memory 연결 (파일로 저장하지 않고 탐색용)
csv_files = sorted(glob.glob("../archive/*.csv"))
csv_files

['../archive\\olist_customers_dataset.csv',
 '../archive\\olist_geolocation_dataset.csv',
 '../archive\\olist_order_items_dataset.csv',
 '../archive\\olist_order_payments_dataset.csv',
 '../archive\\olist_order_reviews_dataset.csv',
 '../archive\\olist_orders_dataset.csv',
 '../archive\\olist_products_dataset.csv',
 '../archive\\olist_sellers_dataset.csv',
 '../archive\\product_category_name_translation.csv']

In [2]:
for path in csv_files:
    info = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").fetchall()
    row_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    print(f"=== {path} (rows: {row_count:,}) ===")
    for col_name, col_type, *_ in info:
        print(f"  {col_name}: {col_type}")
    print()

=== ../archive\olist_customers_dataset.csv (rows: 99,441) ===
  customer_id: VARCHAR
  customer_unique_id: VARCHAR
  customer_zip_code_prefix: VARCHAR
  customer_city: VARCHAR
  customer_state: VARCHAR

=== ../archive\olist_geolocation_dataset.csv (rows: 1,000,163) ===
  geolocation_zip_code_prefix: VARCHAR
  geolocation_lat: DOUBLE
  geolocation_lng: DOUBLE
  geolocation_city: VARCHAR
  geolocation_state: VARCHAR

=== ../archive\olist_order_items_dataset.csv (rows: 112,650) ===
  order_id: VARCHAR
  order_item_id: BIGINT
  product_id: VARCHAR
  seller_id: VARCHAR
  shipping_limit_date: TIMESTAMP
  price: DOUBLE
  freight_value: DOUBLE

=== ../archive\olist_order_payments_dataset.csv (rows: 103,886) ===
  order_id: VARCHAR
  payment_sequential: BIGINT
  payment_type: VARCHAR
  payment_installments: BIGINT
  payment_value: DOUBLE

=== ../archive\olist_order_reviews_dataset.csv (rows: 99,224) ===
  review_id: VARCHAR
  order_id: VARCHAR
  review_score: BIGINT
  review_comment_title: VARC

In [3]:
for path in csv_files:
    print(f"=== {path} ===")
    df = con.execute(f"SELECT * FROM read_csv_auto('{path}') LIMIT 3").df()
    display(df)

=== ../archive\olist_customers_dataset.csv ===


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP


=== ../archive\olist_geolocation_dataset.csv ===


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP


=== ../archive\olist_order_items_dataset.csv ===


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15


=== ../archive\olist_sellers_dataset.csv ===


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


=== ../archive\product_category_name_translation.csv ===


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
